# Notebook 01 — Context Engineering

**You won't change the model once today.** You'll change *what's in the window* and watch the answer change.

Five moves: **STORE → RETRIEVE → CONDENSE → INSERT → ISOLATE.**

Same model, same question — different *context layouts* give visibly different answers. Then three quick measurements: more ≠ better, position matters, isolation beats cramming.

## Setup

`setup()` connects the model and hands back the four things every cell uses: `model`, `ask`, `ntok`, `DATA`. The helpers (`bm25_search`, `chunk_markdown`, `panel`) keep these cells about the *lesson*, not the plumbing. Run it once.

In [ ]:
# ┌─ CONTROL ───────────────────────────────────────────────┐
# │ PROVIDER / MODEL — set ONCE per machine in .env (ARC_PROVIDER/ARC_MODEL).  │
# │ Leave these None to use it; override here only for a one-off switch:      │
# │   "anthropic"  |  "litellm"  |  "ollama"                                  │
# └────────────────────────────────────────────────┘
PROVIDER = None          # None = use .env (ARC_PROVIDER)
MODEL    = None          # None = provider default (ARC_MODEL)

In [ ]:
from pathlib import Path
from textwrap import dedent
import re

from roadshow import setup, chunk_markdown, panel
from roadshow_viz import bars, compare, scatter, stacked

model, ask, ntok, DATA = setup(PROVIDER, MODEL)
INCIDENTS = DATA / "incidents"          # where this lesson's documents live
print("ready · data:", DATA)

## The question we'll answer five ways

One real on-call question, held **constant**. Only the *context* around it changes.

In [ ]:
# The on-call question — identical in every layout below. Only the CONTEXT changes.
QUESTION = "Why did OSS-04 p99 latency spike last night, and what should we do?"

## STORE & RETRIEVE — put the right facts in the window

**Step 1 — load the documents.** One true incident report, plus a pile of distractor ops docs that *look* relevant but aren't.

In [ ]:
# Load the one real incident report + every distractor doc next to it.
TRUE_REPORT   = (INCIDENTS / "storage_incident_2026-04-22.md").read_text()
DISTRACTORS   = sorted((INCIDENTS / "distractors").glob("*.md"))
ALL_DOCS_TEXT = TRUE_REPORT + "\n\n" + "\n\n".join(d.read_text() for d in DISTRACTORS)
print(f"loaded 1 incident report + {len(DISTRACTORS)} distractor docs")

**Step 2 — chunk the docs.** Split each doc into section-sized pieces with `chunk_markdown`. Keeping a symptom and its cause in the same section is what makes retrieval useful.

In [ ]:
# Break every doc into section-sized chunks, each tagged with its source file.
CORPUS = chunk_markdown(TRUE_REPORT, "storage_incident_2026-04-22.md")
for d in DISTRACTORS:
    CORPUS += chunk_markdown(d.read_text(), d.name)
print(f"corpus: {len(CORPUS)} chunks across {1 + len(DISTRACTORS)} docs")

**Step 3 — rank by relevance, built by hand.** "Retrieval" just means *score every chunk against the question and keep the best*. We'll build the scorer ourselves so there's no black box.

Start naive: **count how many of the question's words appear in each chunk.**

In [ ]:
# Naive scorer: how many of the question's words show up in the chunk?
def words(text):
    return re.findall(r"[a-z0-9]+", text.lower())   # lowercase, split into words

def overlap_score(chunk_text, query):
    q = set(words(query))
    return sum(1 for w in words(chunk_text) if w in q)

ranked = sorted(CORPUS, key=lambda c: overlap_score(c["text"], QUESTION), reverse=True)
print("word-overlap top-2 from:", [c["source"] for c in ranked[:2]])

That's crude: every word counts the same, so common words like *"the"*, *"and"*, *"we"* drown out the rare, meaningful one (*"expander"*). And a long chunk wins just by having more words.

**Fix it with TF-IDF** — the idea behind every real search engine:
- **TF** (term frequency): how often a query word appears *in this chunk*.
- **IDF** (inverse document frequency): how *rare* that word is across *all* chunks. A word in every chunk is worthless; a word in one chunk is gold.

Score = Σ (TF × IDF) over the question's words. Let's compute it by hand.

In [ ]:
import math
from collections import Counter

# IDF — measured ONCE over the whole corpus: rare words score high, common low.
N = len(CORPUS)
doc_freq = Counter()                       # in how many chunks does each word appear?
for c in CORPUS:
    for w in set(words(c["text"])):
        doc_freq[w] += 1

def idf(word):
    # log so the curve is gentle; +1s keep it defined for every word.
    return math.log(1 + N / (1 + doc_freq[word]))

def tfidf_score(chunk_text, query):
    tf = Counter(words(chunk_text))        # term frequency: word -> count in chunk
    return sum(tf[w] * idf(w) for w in set(words(query)))

# Rank every chunk by TF-IDF, keep the best 2 — this is our "retrieval".
ranked = sorted(CORPUS, key=lambda c: tfidf_score(c["text"], QUESTION), reverse=True)
TOP_CHUNKS = ranked[:2]
print("TF-IDF top-2 from:", [c["source"] for c in TOP_CHUNKS])

# Peek at the rare words doing the work (highest IDF among the question's terms):
q_terms = sorted(set(words(QUESTION)), key=idf, reverse=True)[:5]
print("most distinctive question words:", [(w, round(idf(w), 2)) for w in q_terms])

**From TF-IDF to BM25.** Production search (Elasticsearch, Lucene) uses **BM25**, which is TF-IDF plus two refinements:
1. **Saturation** — the 10th occurrence of a word adds far less than the 2nd (one chunk can't dominate by repeating a term).
2. **Length normalization** — a long chunk doesn't win just for being long.

Same TF × IDF core you just built — only tuned. For this lesson, TF-IDF already pulls the right chunk.

## Four context layouts — built and run one at a time

We answer the **same** `QUESTION` four ways, changing only what's in the window. Each layout gets its own cell below — run them in order and **scroll up to compare** the answers.

- **minimal** — just the question (raw-chat baseline)
- **stuffed** — question + the *entire* corpus ("give it everything")
- **retrieved** — question + only the **2 BM25 chunks**
- **laid_out** — those chunks + system framing + a scratch budget

In [ ]:
# We keep each answer + its input size so the charts later can compare them.
RESULTS = {}

# The "laid out" system prompt: a role, strict instructions, one reversible action.
SYS_LAID = dedent("""
    You are a senior Lustre storage operator at an exascale HPC facility.
    Answer the on-call question using ONLY the provided context.
    Name the affected system, state the root cause, and propose ONE reversible
    action. Be concise. If the context does not contain the cause, say so.
""").strip()

async def run_layout(name, system, user):
    """Send ONE layout to the model, show the reply, and remember it for the
    charts below. `system` may be None (no framing); `user` is the full prompt."""
    answer = await ask(user, system=system)
    RESULTS[name] = {"answer": answer,
                     "in_tokens": ntok((system or "") + user),
                     "out_tokens": ntok(answer)}
    panel(answer, title=f"{name.upper()}  ·  input ≈ {RESULTS[name]['in_tokens']} tokens",
          style="green" if name in ("retrieved", "laid_out") else "yellow")

**Layout 1 — minimal.** Just the question. No facts to work with.

In [ ]:
# MINIMAL — the question alone. The baseline most people start with.
await run_layout("minimal", None, QUESTION)

**Layout 2 — stuffed.** The question plus the *entire* corpus. Naive "give it everything" — watch it latch onto a distractor.

In [ ]:
# STUFFED — dump every document into the window and hope the model finds the needle.
stuffed_user = f"Here is everything we have. {QUESTION}\n\n<corpus>\n{ALL_DOCS_TEXT}\n</corpus>"
await run_layout("stuffed", None, stuffed_user)

**Layout 3 — retrieved.** Only the 2 BM25 chunks. A fraction of the tokens.

In [ ]:
# RETRIEVED — give the model ONLY the 2 chunks BM25 ranked highest.
context = "\n\n".join(f"[{c['source']}]\n{c['text']}" for c in TOP_CHUNKS)
retrieved_user = f"Context:\n{context}\n\nQuestion: {QUESTION}"
await run_layout("retrieved", None, retrieved_user)

**Layout 4 — laid_out.** The same retrieved chunks, but with system framing, the question in tags, and room to think.

In [ ]:
# LAID_OUT — same facts as "retrieved", but framed: a role, the question in
# tags, the context in tags, and an explicit "plan first" instruction.
laid_user = dedent(f"""
    <question>
    {QUESTION}
    </question>

    <context>
    {context}
    </context>

    Write a 3-line scratch plan first (cause, reversible?, action), then give the operator answer.
""").strip()
await run_layout("laid_out", SYS_LAID, laid_user)

### Name the failure mode

- **minimal** is vague — no facts.
- **stuffed** often grabs a *distractor* — confusion + distraction, live.
- **retrieved** is sharp: slot-42 backplane power excursion → OST-0042 offline.
- **laid_out** is sharp *and* structured.

The exact failure varies run-to-run — that variability *is* the lesson.

### Visual — where each layout spends the window

Watch `stuffed` dwarf the rest while `retrieved` and `laid_out` stay lean.

> Token counts here use tiktoken's `cl100k_base`, which is OpenAI's tokenizer — an approximation when the model answering is Anthropic or Ollama. Close enough for these budgeting visuals, not an exact count for the model in use.

In [ ]:
# Split each layout's input into segments so the token chart is honest.
def budget(name):
    payload = (ntok(ALL_DOCS_TEXT) if name == "stuffed"
               else ntok(context) if name in ("retrieved", "laid_out") else 0)
    sys_tok = ntok(SYS_LAID) if name == "laid_out" else 0
    return {"system": sys_tok, "question": ntok(QUESTION), "payload": payload}

BUDGET = {name: budget(name) for name in ["minimal", "stuffed", "retrieved", "laid_out"]}
stacked(BUDGET, title="Token budget per context layout", ylabel="input tokens")

## Measure 1 — cost vs quality

We score quality on **specifics only the right context supplies**: the failed component (OST-0042 / slot 42), the real cause (SAS-expander power excursion), and a reversible action. (We don't credit "OSS-04" — it's already in the question.)

In [ ]:
def _flat(text):
    """Lowercase and turn every run of punctuation into a single space, padded
    with spaces — so 'OST-0042', 'OST 0042' and 'ost0042' all match the same way."""
    return " " + re.sub(r"[^a-z0-9]+", " ", (text or "").lower()).strip() + " "

def rubric_score(answer):
    """A transparent 0–3 score. +1 for naming the failed COMPONENT, +1 for the
    real CAUSE, +1 for a reversible ACTION. Matching is lenient (synonyms,
    spacing) so a small model that genuinely found the answer still gets credit."""
    a = _flat(answer)
    def has(*terms): return any(t in a for t in terms)
    score = 0
    if has(" ost 0042 ", " ost0042 ", " ost 42 ", " slot 42 ", " nvme8n1 ",
           " drive 42 ", " disk 42 "):                         # failed COMPONENT
        score += 1
    if has(" backplane ", " expander ", " sas ", " power excursion ",
           " power fault ", " power spike ", " voltage "):     # real CAUSE
        score += 1
    if has(" evict ", " evicted ", " auto evict ", " rma ", " replace ",
           " threshold ", " offline ", " drain ", " failover ",
           " route around ", " disable "):                     # reversible ACTION
        score += 1
    return score

**Plot cost vs quality.** `retrieved` and `laid_out` reach `stuffed`'s quality at a fraction of the tokens.

In [ ]:
# Score every layout's answer with the rubric above, then plot tokens vs quality.
QUALITY = {name: rubric_score(RESULTS[name]["answer"]) for name in RESULTS}
COST_QUALITY = {name: (RESULTS[name]["in_tokens"], QUALITY[name]) for name in QUALITY}
print("quality (0-3):", QUALITY)
scatter(COST_QUALITY, title="Cost vs quality: more context ≠ better",
        xlabel="input tokens  (cost →)", ylabel="rubric quality (0-3)")

### Why "more" lost — attention is finite

A transformer's attention weights are **normalized per token** (softmax across the window), so adding more tokens doesn't shrink some fixed pool — but it does dilute the signal: the few tokens that matter now compete with thousands of irrelevant ones inside a finite context window. What counts is **relevance per token**, not token count.

## INSERT — does *position* alone change the answer? ("lost in the middle")

Now we hold the **content** constant and change only the **order**. We bury the one true-cause chunk among distractors and slide it from the **start** of the window to the **end**, asking the same question each time. Does the model still find the cause?

> **Model-dependent.** A strong model often finds it everywhere (a flat line near 1.0 = good). Weaker or longer-context models dip in the middle (Liu et al. 2024). We **measure** it live.

In [ ]:
# The "needle" = the one chunk that actually contains the root cause.
NEEDLE = next(c for c in CORPUS
              if "backplane fault" in c["text"].lower() or "expander" in c["text"].lower())

# One distractor chunk per distractor doc — the "haystack" we hide the needle in.
DISTRACTOR_CHUNKS = []
for d in DISTRACTORS:
    paras = [p.strip() for p in re.split(r"\n\s*\n", d.read_text()) if p.strip()]
    DISTRACTOR_CHUNKS.append({"source": d.name, "text": paras[-1] if paras else d.read_text()})
print(f"needle from: {NEEDLE['source']}  ·  {len(DISTRACTOR_CHUNKS)} distractors")

In [ ]:
# Insert the needle at a chosen spot among the distractors, glued into one window.
POSITION_AT = {"start": lambda n: 0, "q1": lambda n: n // 4, "middle": lambda n: n // 2,
               "q3": lambda n: (3 * n) // 4, "end": lambda n: n}

def build_window(position):
    """Place the needle at `position` among the distractors; return the text window."""
    idx = POSITION_AT[position](len(DISTRACTOR_CHUNKS))
    chunks = DISTRACTOR_CHUNKS[:idx] + [NEEDLE] + DISTRACTOR_CHUNKS[idx:]
    return "\n\n".join(f"[{c['source']}]\n{c['text']}" for c in chunks)

def found_cause(answer):
    """True if the answer names the real cause (lenient term match)."""
    a = _flat(answer)
    return any(t in a for t in [" backplane ", " expander ", " sas ", " power excursion ",
               " power fault ", " slot 42 ", " ost 0042 ", " ost0042 ", " ost 42 "])

**Run the sweep.** Five positions × `TRIALS` repeats each. We print, per position, how often the model found the cause — then chart it.

In [ ]:
TRIALS = 3   # repeats per position (more = less noise, more calls)
SYS_NEEDLE = ("You are a storage operator. Using ONLY the context, answer in one "
              "sentence: what was the root cause of the OSS-04 latency spike?")

POSITIONS = ["start", "q1", "middle", "q3", "end"]
recall = {}
for pos in POSITIONS:
    hits = 0
    for _ in range(TRIALS):
        answer = await ask(f"<context>\n{build_window(pos)}\n</context>\n\nQuestion: what caused it?",
                           system=SYS_NEEDLE)
        hits += found_cause(answer)
    recall[pos] = hits / TRIALS
    print(f"  needle at {pos:<6} → found {hits}/{TRIALS}  (recall {recall[pos]:.2f})")

bars(recall, title="Lost in the middle: did the model find the cause?",
     xlabel="needle position in the window", ylabel="recall (0–1)", fmt="{:.2f}")

## CONDENSE — shrink the window, keep the decision

The incident bridge log is ~40 turns. We condense it two ways — a careful handoff and a deliberately terse one — and check whether the key **decision** survives. A bad summary looks authoritative but quietly drops the key fact.

In [ ]:
def preserved(summary):
    """Did the summary keep the DECISION *and* what it acted on? We check for an
    action word (evict / offline / drain ...) AND the object it applied to
    (the OST id). Both must survive, or the handoff lost the point."""
    s = _flat(summary)
    action = any(t in s for t in [" evict ", " evicted ", " auto evict ", " offline ",
                                  " drain ", " removed ", " disable ", " took offline "])
    object_ = any(t in s for t in [" ost 0042 ", " ost0042 ", " ost 42 ", " slot 42 ", " the ost "])
    return action and object_

**Two summaries, one log.** One careful handoff that must keep the decision; one deliberately lossy.

In [ ]:
LOG = (INCIDENTS / "oss04_running_log.md").read_text()

# A good handoff is TOLD to keep the cause, the decision, the outcome, the follow-ups.
GOOD_SYS = dedent("""
    Condense this incident bridge log into a 5-line state handoff. MUST preserve:
    the root cause, the key DECISION taken (what was evicted and when), the
    recovery outcome, and the follow-up actions. No fluff.
""").strip()
# A lossy summary is told to cover ONLY customer impact — and drops the decision.
BAD_SYS = "Summarize this log in ONE short sentence about customer impact only. Do not mention internal actions."

good_summary = await ask(f"<log>\n{LOG}\n</log>", system=GOOD_SYS)
bad_summary  = await ask(f"<log>\n{LOG}\n</log>", system=BAD_SYS)

panel(good_summary, title=f"GOOD handoff · {ntok(good_summary)} tok · decision kept = {preserved(good_summary)}", style="green")
panel(bad_summary,  title=f"BAD summary · {ntok(bad_summary)} tok · decision kept = {preserved(bad_summary)}", style="red")

**Plot the compression.** Same shrink, very different value — the lossy summary dropped the decision.

In [ ]:
COMPRESSION = {"full log": ntok(LOG), "good summary": ntok(good_summary), "bad summary": ntok(bad_summary)}
compare(COMPRESSION, title="Compression: shrink tokens, keep the decision?", ylabel="tokens")

## ISOLATE — don't cram it all in one window

Sometimes the fix isn't a better summary — it's *not putting everything in one window*. Route each noisy doc to its **own scoped call** that returns a one-line finding, then a final call sees only the **findings** plus the question. (This is what sub-agents do — Notebook 06 builds the real version.)

In [ ]:
# Every doc as (name, text).
ALL_DOC_TEXTS = [("storage_incident_2026-04-22.md", TRUE_REPORT)] + \
                [(d.name, d.read_text()) for d in DISTRACTORS]

# APPROACH A — MONOLITH: dump every doc + the question into ONE window.
mono_user = ("Using all the context, answer: " + QUESTION + "\n\n<corpus>\n" +
             "\n\n".join(f"[{n}]\n{t}" for n, t in ALL_DOC_TEXTS) + "\n</corpus>")
mono_answer = await ask(mono_user, system=SYS_LAID)
mono_peak = ntok((SYS_LAID or "") + mono_user)
panel(mono_answer, title=f"MONOLITH · peak window ≈ {mono_peak} tokens", style="yellow")

In [ ]:
# APPROACH B — ISOLATE: one scoped call per doc → a one-line finding. The final
# call sees ONLY the findings, never the raw docs. Note the peak window stays small.
SCOPE_SYS = ("Read ONE ops doc. In one sentence, state whether it explains the OSS-04 "
             "latency spike, and if so the cause. If irrelevant, say 'not relevant'. Be terse.")

findings, peak_scoped = [], 0
for name, text in ALL_DOC_TEXTS:
    scoped_user = f"<doc name='{name}'>\n{text}\n</doc>\n\nQuestion: {QUESTION}"
    peak_scoped = max(peak_scoped, ntok(SCOPE_SYS + scoped_user))
    finding = await ask(scoped_user, system=SCOPE_SYS)
    findings.append(f"[{name}] {finding.strip()}")

final_user = "Findings from scoped reviewers (raw docs NOT included):\n" + "\n".join(findings) + f"\n\nQuestion: {QUESTION}"
iso_answer = await ask(final_user, system=SYS_LAID)
iso_peak = max(peak_scoped, ntok((SYS_LAID or "") + final_user))
panel(iso_answer, title=f"ISOLATED · peak window ≈ {iso_peak} tokens", style="green")

**Compare the peak window** each approach needed — and the quality.

In [ ]:
compare({"monolith": mono_peak, "isolated": iso_peak},
        title="Peak window size: monolith vs isolated", ylabel="tokens")
compare({"monolith": rubric_score(mono_answer), "isolated": rubric_score(iso_answer)},
        title="Answer quality: monolith vs isolated", ylabel="rubric (0-3)", fmt="{:.0f}")

## The five moves you just ran

| Move | What you did | Field term | Deepened in |
|---|---|---|---|
| **STORE** | kept the corpus on disk, not in the prompt | *Write* | Harness / Identity |
| **RETRIEVE** | BM25 top-2 chunk selection | *Select* | Skills |
| **CONDENSE** | 40-turn log → 5-line handoff | *Compress* | Loop / Adaptation |
| **INSERT** | `laid_out` framing + the position experiment | *(every prompt)* | every notebook |
| **ISOLATE** | scoped per-doc calls → findings-only synthesis | *Isolate* | Workflows / sub-agents |

You changed the answer **five times without touching the model.**

## From moves to agents

You ran the five moves by hand. In an **agent**, the model runs them on *itself* every turn — it fills its own window, passes findings not transcripts, holds context as an inspectable state object, and survives long runs by planning, isolating, offloading, and compacting. Same five verbs, sharpened — that's the rest of the day.

## ✅ TODO — your own layout

Drop in one of *your* questions and one of *your* docs. Pick a `LAYOUT`, run, and try to **beat `stuffed` with `retrieved`** on your own data.

In [ ]:
# ✅ TODO — edit these three, then run.
MY_QUESTION = "TODO — a real question from your work (an incident, a CVE, a ticket)."
MY_DOC_PATH = INCIDENTS / "storage_incident_2026-04-22.md"   # TODO — point at YOUR doc
MY_LAYOUT   = "retrieved"   # "minimal" | "stuffed" | "retrieved" | "laid_out"

my_text   = Path(MY_DOC_PATH).read_text()
my_chunks = chunk_markdown(my_text, Path(MY_DOC_PATH).name)

if MY_LAYOUT == "minimal":
    my_sys, my_usr = None, MY_QUESTION
elif MY_LAYOUT == "stuffed":
    my_sys, my_usr = None, f"{MY_QUESTION}\n\n<corpus>\n{my_text}\n</corpus>"
else:
    # reuse the TF-IDF scorer we built above to pick the 2 best chunks
    top = sorted(my_chunks, key=lambda c: tfidf_score(c["text"], MY_QUESTION), reverse=True)[:2]
    payload = "\n\n".join(f"[{c['source']}]\n{c['text']}" for c in top)
    my_sys = SYS_LAID if MY_LAYOUT == "laid_out" else None
    my_usr = f"<context>\n{payload}\n</context>\n\nQuestion: {MY_QUESTION}"

my_answer = await ask(my_usr, system=my_sys)
panel(my_answer, title=f"LAYOUT={MY_LAYOUT} · input ≈ {ntok((my_sys or '') + my_usr)} tokens")

## Takeaway

> You changed the answer five times without touching the model. **That's the whole job.** Next: the agentic loop — the mechanism that decides *when* to fetch context.